# Canonical pair: Beauty + ML-1M

Five models, one full-catalog evaluation contract per dataset:

- SASRec + FullCE
- SASRec + SS256
- LiGR + FullCE
- eSASRec
- **Sparse Walker + FullCE**

The four already-trained SASRec/LiGR checkpoints are **re-evaluated** through the same current evaluator used for Walker. The script records a protocol fingerprint derived from the actual split/evaluator source.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!rm -rf /content/Sparsewalker
!git clone -q https://github.com/hanialshater/Sparsewalker-.git /content/Sparsewalker
%cd /content/Sparsewalker
!pip -q install -e .
import torch
print('torch', torch.__version__)
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 1. Beauty

This should mostly re-evaluate the four completed 2×2 checkpoints and spend its time training Walker.

In [ ]:
OUT='/content/drive/MyDrive/sparsewalker_canonical_pair'
BASE='/content/drive/MyDrive/sparsewalker_esasrec_2x2'
!python benchmarks/run_canonical_pair.py \
  --dataset beauty --seed 42 \
  --baseline-root "$BASE" --output-dir "$OUT" \
  --eval-batch-size 1024

## 2. ML-1M

Same evaluator contract and five-model matrix. Walker uses max_len=200 and the same K=8 / degree=4 / 2-hop state geometry.

In [ ]:
!python benchmarks/run_canonical_pair.py \
  --dataset ml1m --seed 42 \
  --baseline-root "$BASE" --output-dir "$OUT" \
  --eval-batch-size 1024

## 3. Joint conclusion table

In [ ]:
import pandas as pd
from pathlib import Path
root=Path(OUT)
frames=[]
for ds in ['beauty','ml1m']:
    x=pd.read_csv(root/ds/'seed42'/'summary.csv')
    x.insert(0,'dataset',ds)
    frames.append(x)
all_results=pd.concat(frames,ignore_index=True)
display(all_results[['dataset','cell','NDCG@10','HR@10','MRR@10','params','protocol_fingerprint']])
pivot=all_results.pivot(index='cell',columns='dataset',values='NDCG@10')
display(pivot.sort_values('beauty',ascending=False))
print('All rows within each dataset must share exactly one protocol_fingerprint:')
print(all_results.groupby('dataset').protocol_fingerprint.nunique())